## Exercise 2

In this exercise, we will predict the two income classes in the adult dataset (The file "adult.csv" is also on Moodle). 

Answer the following questions:
1. Clean the `income` variable such that it has only two values (for instance by using `str.replace` to remover the `.`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Load dataset
df = pd.read_csv("adult.csv")

# Remove '.' from income labels
df["income"] = df["income"].str.replace(".", "", regex=False)

# Check values
df["income"].value_counts()


income
<=50K    37155
>50K     11687
Name: count, dtype: int64

2. Select as set of minimum two feature variables you want to use to predict `income`. Do the necessary transformation of these variables.


In [ ]:

# Select features and target
X = df[["age", "education-num", "sex"]]
y = df["income"]

# Convert categorical variable to dummy
X = pd.get_dummies(X, columns=["sex"], drop_first=True)



3. Create X and y dataset and split the datasets into training and testing sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


4. Train a KNN classifier to predict the variable `income` based on the feature variables selected in 2 - try out some different Ks 


In [ ]:
k_values = [3, 5, 7, 11, 15]
knn_results = {}

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    
    knn_results[k] = accuracy_score(y_test, y_pred)

knn_results

best_k = max(knn_results, key=knn_results.get)
best_k


15

5. Train a logistic regression classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the KNN classifier.


In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)

log_acc = accuracy_score(y_test, y_pred_log)
log_acc


0.7975227761285699

6. Train a decision tree classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_acc


0.8013102671716654

7. Train a random forest classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_acc


0.8015149964172382

8. Train a AdaBoost classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.


In [ ]:
ada = AdaBoostClassifier(
    n_estimators=200,
    random_state=42
)
ada.fit(X_train, y_train)

y_pred_ada = ada.predict(X_test)

ada_acc = accuracy_score(y_test, y_pred_ada)
ada_acc

results = pd.DataFrame({
    "Model": [
        f"KNN (K={best_k})",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "AdaBoost"
    ],
    "Accuracy": [
        knn_results[best_k],
        log_acc,
        dt_acc,
        rf_acc,
        ada_acc
    ]
})

results.sort_values(by="Accuracy", ascending=False)



0.8058143105742656

9. (Optional) Can you improve any of the models by balancing the `income` variable?

In [ ]:
log_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
log_balanced.fit(X_train_scaled, y_train)

y_pred_bal = log_balanced.predict(X_test_scaled)

print("Balanced Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_bal))
print("Precision:", precision_score(y_test, y_pred_bal, pos_label=">50K"))
print("Recall:", recall_score(y_test, y_pred_bal, pos_label=">50K"))
print("F1:", f1_score(y_test, y_pred_bal, pos_label=">50K"))


Balanced Logistic Regression
Accuracy: 0.7175760057324189
Precision: 0.44376168848517233
Recall: 0.7104362703165098
F1: 0.5462917283341555
